In [4]:
import os
import time
import traceback
import nbformat
from nbconvert.preprocessors import ExecutePreprocessor, CellExecutionError


# CONFIG

BASE_DIR = os.getcwd()   

NOTEBOOKS = [
    os.path.join(BASE_DIR, "Mass", "import mass script v3.ipynb"),
    os.path.join(BASE_DIR, "Dimensional", "import dimensional.ipynb"),
    os.path.join(BASE_DIR, "Pressure & Force", "pressure and force import script.ipynb"),
    os.path.join(BASE_DIR, "Temp & Hum", "temp_hum_importer.ipynb"),
    os.path.join(BASE_DIR, "Acoustics", "acoustics_importer.ipynb"),
]

KERNEL_NAME = "python3"
TIMEOUT = None
STOP_ON_ERROR = True


# RUNNER

ep = ExecutePreprocessor(timeout=TIMEOUT, kernel_name=KERNEL_NAME)

results = []
start_all = time.time()

for nb_path in NOTEBOOKS:
    nb_name = os.path.basename(nb_path)
    nb_dir = os.path.dirname(nb_path)

    print("\n" + "=" * 80)
    print(f"RUNNING: {nb_name}")
    print(f"FOLDER : {nb_dir}")
    print("=" * 80)

    if not os.path.exists(nb_path):
        print(f"❌ File not found: {nb_path}")
        results.append((nb_name, "FILE NOT FOUND"))
        if STOP_ON_ERROR:
            break
        continue

    t0 = time.time()

    try:
        with open(nb_path, "r", encoding="utf-8") as f:
            nb = nbformat.read(f, as_version=4)

        
        # execute notebook inside ITS OWN folder
        ep.preprocess(nb, {"metadata": {"path": nb_dir}})

        executed_name = nb_name.replace(".ipynb", "_EXECUTED.ipynb")
        executed_path = os.path.join(nb_dir, executed_name)

        with open(executed_path, "w", encoding="utf-8") as f:
            nbformat.write(nb, f)

        dt = time.time() - t0
        print(f"✅ Finished: {nb_name}")
        print(f"💾 Saved executed copy: {executed_path}")
        print(f"⏱️ Time: {dt/60:.2f} min")
        results.append((nb_name, f"OK ({dt/60:.2f} min)"))

    except CellExecutionError as e:
        dt = time.time() - t0
        print(f"❌ Notebook failed: {nb_name}")
        print(f"⏱️ Time before failure: {dt/60:.2f} min")
        print("\n--- ERROR ---")
        print(str(e))
        print("-------------")
        results.append((nb_name, f"FAILED ({dt/60:.2f} min)"))
        if STOP_ON_ERROR:
            break

    except Exception as e:
        dt = time.time() - t0
        print(f"❌ Unexpected error in: {nb_name}")
        print(f"⏱️ Time before failure: {dt/60:.2f} min")
        traceback.print_exc()
        results.append((nb_name, f"FAILED ({dt/60:.2f} min)"))
        if STOP_ON_ERROR:
            break


# SUMMARY

total_dt = time.time() - start_all

print("\n" + "=" * 80)
print("MASTER RUN SUMMARY")
print("=" * 80)
for nb_name, status in results:
    print(f"{nb_name}: {status}")

print(f"\nTotal elapsed time: {total_dt/60:.2f} min")


RUNNING: import mass script v3.ipynb
FOLDER : C:\Users\ambrasasm\db_project\Mass
✅ Finished: import mass script v3.ipynb
💾 Saved executed copy: C:\Users\ambrasasm\db_project\Mass\import mass script v3_EXECUTED.ipynb
⏱️ Time: 0.05 min

RUNNING: import dimensional.ipynb
FOLDER : C:\Users\ambrasasm\db_project\Dimensional
✅ Finished: import dimensional.ipynb
💾 Saved executed copy: C:\Users\ambrasasm\db_project\Dimensional\import dimensional_EXECUTED.ipynb
⏱️ Time: 0.06 min

RUNNING: pressure and force import script.ipynb
FOLDER : C:\Users\ambrasasm\db_project\Pressure & Force
✅ Finished: pressure and force import script.ipynb
💾 Saved executed copy: C:\Users\ambrasasm\db_project\Pressure & Force\pressure and force import script_EXECUTED.ipynb
⏱️ Time: 6.54 min

RUNNING: temp_hum_importer.ipynb
FOLDER : C:\Users\ambrasasm\db_project\Temp & Hum
✅ Finished: temp_hum_importer.ipynb
💾 Saved executed copy: C:\Users\ambrasasm\db_project\Temp & Hum\temp_hum_importer_EXECUTED.ipynb
⏱️ Time: 0.89 mi